# Brainrot Transformer 
This is my implementation of Andrej Karpathy's "Building GPT from Scratch". All credits go to the original author. This notebook involves making the **Transformer** architecture from scratch but with my own twist. Instead of training it on the Tiny Shakespeare Dataset, this is trained on **Brainrot Corpus** which was generated by me using Claude and involves no web scraping. Both datatsets are similar in file size (1.1MB) and character count (1 million) for continuity purposes.

## Setting up and Inspecting the Dataset

In [2]:
# Opening and reading our dataset
with open('brainrot_corpus.txt','r') as f:
    text = f.read()

In [3]:
print("Length of dataset in terms of characters ",len(text))

Length of dataset in terms of characters  1115384


In [4]:
# Seeing the first couple characters
print(text[:1000])

CRASHOUT KAREN:
Bruh, it's giving rizz, Grimace Shake Guy
bro is cooked fr fr.
RIP my the AI slop, Sigma said six seven for
no reason with a rizz, don't leave me dry.

CHAT:
Rizzler, why you actin like a whole the
bazooka? don't leave me dry.

CHOPPELGANGER:
Yo, the skibidi is crazy, Chat lost all his
aura again, clock it - fr!
Deadass, aura minus 1000. Chat really yeeted
the bazooka in front of everybody.

CHOPPELGANGER:
Bro got rejected by the ohio energy and
pulled up with the the cap, rizz level 100.
Ngl, on god. Chat really farmed aura for no
reason in front of everybody.

MEWING MIKE:
Hold on, it's giving the doi doi doi, Mewing
Mike this is so ohio fr fr.
Sigma'S Little Brother is mewing in the
mirror and I said this is so ohio, that's
the the sauce.
Fam, did you hear Ohio Npc is goated with
the sauce? ohio final boss.

CROCODILO:
Chat said brainrot vibes and then farmed
aura for no reason.

GYATT GURL:
Sigma said aura at 0% and then is goated
with the sauce.
Hold on, Ohio Npc p

In [5]:
# to see all the unique characters that occur in this text
chars = list(set(text))
print(''.join(chars))
print(len(chars))

ETDba.1#wXHf?hO%Apx6Z2'KyVLSc7sg
IPe!Gr,0onWjBR-FulMYCdUNkvq:t imz
66


## Tokenization
The next step is to convert text to a sequence of integers. To keep things simple we are going to use character level tokenization meaning each character is represented by a different integer. The size of our vocabulary (ie number of unique tokens) will be the number of unique characters which is 66. Character tokenization has small vocabulary sizes but long sequences of tokens to represent a line as compared to sub-word encoding used by latest LLMs which convert a part of a word into a token (integer) and hence have a huge vocabulary but smaller sequences to represent the same string.

In [8]:
# making the vocabulary and the encode-decode functions
vocab_size = len(chars)
char_to_int = {c:i for i,c in enumerate(chars)}
int_to_char = {i:c for i,c in enumerate(chars)}
encode = lambda s: [char_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_char[i] for i in l)
eg = encode("hello")
print("encoding the sentence 'hello' ",eg)
print("decoding the same ",decode(eg))

encoding the sentence 'hello'  [13, 35, 50, 50, 41]
decoding the same  hello


In [11]:
# now encoding/tokenizing the entire dataset into a pytorch tensor
import torch
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape,data.dtype)
print(data[:1000])

torch.Size([1115384]) torch.int64
tensor([53, 46, 16, 27, 10, 14, 55,  1, 62, 23, 16, 46,  0, 56, 60, 32, 45, 38,
        49, 13, 39, 62, 63, 61, 22, 30, 62, 31, 63, 58, 63, 42, 31, 62, 38, 63,
        65, 65, 39, 62, 37, 38, 63, 64,  4, 28, 35, 62, 27, 13,  4, 57, 35, 62,
        37, 49, 24, 32,  3, 38, 41, 62, 63, 30, 62, 28, 41, 41, 57, 35, 54, 62,
        11, 38, 62, 11, 38,  5, 32, 46, 33, 34, 62, 64, 24, 62, 61, 13, 35, 62,
        16, 33, 62, 30, 50, 41, 17, 39, 62, 27, 63, 31, 64,  4, 62, 30,  4, 63,
        54, 62, 30, 63, 18, 62, 30, 35, 58, 35, 42, 62, 11, 41, 38, 32, 42, 41,
        62, 38, 35,  4, 30, 41, 42, 62,  8, 63, 61, 13, 62,  4, 62, 38, 63, 65,
        65, 39, 62, 54, 41, 42, 22, 61, 62, 50, 35,  4, 58, 35, 62, 64, 35, 62,
        54, 38, 24,  5, 32, 32, 53, 10, 16,  1, 60, 32, 46, 63, 65, 65, 50, 35,
        38, 39, 62,  8, 13, 24, 62, 24, 41, 49, 62,  4, 28, 61, 63, 42, 62, 50,
        63, 57, 35, 62,  4, 62,  8, 13, 41, 50, 35, 62, 61, 13, 35, 32,  3,  4,
      

In [13]:
# finally splitting the data into training and validation, doing a 90% split
split = int(0.9*len(data))
train_data = data[:split]
val_data = data[split:]

## Batching
After tokenization comes the process of batching. While training the transformer we don't give it the entire dataset just once. Since it is learning to predict the next token (in this case the next character) we randomly take blocks of data of a fixed size and then provide the transformer multiple examples of the block to predict the next token (given only the first token of the block predict the second, given the first 2 predict the 3rd and so on...). A group of multiple blocks is called a batch.

In [15]:
# demonstration of how a single batch can be used to provide multiple training examples
block_size = 8
sample = data[:batch_size]
target = data[1:batch_size+1]
for i in range(batch_size):
    context = sample[:i+1]
    response = target[i]
    print("When context is ",context," the target is ",response)

When context is  tensor([53])  the target is  tensor(46)
When context is  tensor([53, 46])  the target is  tensor(16)
When context is  tensor([53, 46, 16])  the target is  tensor(27)
When context is  tensor([53, 46, 16, 27])  the target is  tensor(10)
When context is  tensor([53, 46, 16, 27, 10])  the target is  tensor(14)
When context is  tensor([53, 46, 16, 27, 10, 14])  the target is  tensor(55)
When context is  tensor([53, 46, 16, 27, 10, 14, 55])  the target is  tensor(1)
When context is  tensor([53, 46, 16, 27, 10, 14, 55,  1])  the target is  tensor(62)


In [18]:
# when training a transformer we use multiple bactches called blocks
block_size = 8 # size of a sequence of data
batch_size = 4 # number of blocks
torch.manual_seed(1337)
def get_batch(split):
    data = train_data if split=='train' else val_data
    ix = torch.randint(len(data) - block_size,(batch_size,))
    x = torch.stack([data[i:block_size+i] for i in ix])
    y = torch.stack([data[i+1:block_size+i+1] for i in ix])
    return x,y
x,y = get_batch('train')
print("size of x ",x.shape)
print(x)
print("size of y ",y.shape)
print(y)
# total number of training examples given by this entire batch = 4 * 8 = 32
for i in range(batch_size):
    for j in range(block_size):
        context = x[i][:j+1]
        target = y[i][j]
        print("When context is ",context," the target is ",target)

size of x  torch.Size([4, 8])
tensor([[61, 32,  4, 42, 54, 62, 61, 13],
        [41, 41, 57, 62, 61, 13, 35, 32],
        [62, 64, 41, 30, 59, 49, 63, 61],
        [62, 61, 13, 35, 62, 61, 13, 35]])
size of y  torch.Size([4, 8])
tensor([[32,  4, 42, 54, 62, 61, 13, 35],
        [41, 57, 62, 61, 13, 35, 32, 11],
        [64, 41, 30, 59, 49, 63, 61, 41],
        [61, 13, 35, 62, 61, 13, 35, 62]])
When context is  tensor([61])  the target is  tensor(32)
When context is  tensor([61, 32])  the target is  tensor(4)
When context is  tensor([61, 32,  4])  the target is  tensor(42)
When context is  tensor([61, 32,  4, 42])  the target is  tensor(54)
When context is  tensor([61, 32,  4, 42, 54])  the target is  tensor(62)
When context is  tensor([61, 32,  4, 42, 54, 62])  the target is  tensor(61)
When context is  tensor([61, 32,  4, 42, 54, 62, 61])  the target is  tensor(13)
When context is  tensor([61, 32,  4, 42, 54, 62, 61, 13])  the target is  tensor(35)
When context is  tensor([41])  the 

## Simple Langauge Modeling: Bigrams
Before diving into transformers, let's make the simplest language model possible: Bigrams, which are just fancy lookup tables.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class Bigram(nn.Module): # inherits stuff from nn.Module
    